# Nonparametric Chernoff Information Estimation

This notebook estimates the **Chernoff information** between two distributions $P$ and $Q$ directly from samples, using $k$-nearest-neighbour (kNN) density-ratio estimates.

The Chernoff $\alpha$-coefficient and Chernoff information are

$$C_s(P,Q) = -\log \int p(x)^s\, q(x)^{1-s}\, dx, \qquad C(P,Q) = \max_{s \in (0,1)} C_s(P,Q) = C_{s^*}(P,Q).$$

$C_s$ is concave in $s$, so the optimal exponent $s^*$ can be found by bisection on the sign of its derivative (Nielsen's bisection scheme).

**Sections**
1. Synthetic data: estimator sanity check, bisection trace, and convergence vs. sample size
2. MNIST: Chernoff information between PCA-projected digit classes
3. Dimensionality scaling: estimator error vs. sample size for increasing dimension

In [ ]:
import torch
from chernoff_utils import get_dataset, DATASETS, kth_dists
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import gammaln, digamma
from scipy.stats import norm
import tqdm

print(f"Available Datasets: {list(DATASETS.keys())}")

## Synthetic data

Draw samples from a synthetic pair $(P, Q)$ for which the true optimal exponent $s^*$ and true Chernoff information $C(P,Q)$ are known in closed form (see `chernoff_data.py`).

In [ ]:
dims = [2,4,8,16]
# Returns samples P, Q plus the ground-truth optimal exponent s* and Chernoff information
dataset_kwargs = {"name": "symmetric_truncnorm", "delta": 0.1, "sigma": 0.1}
P, Q, true_s, true_c = get_dataset(**dataset_kwargs)

### kNN estimators of $C_s$ and its derivative

For each $x_i \sim P$, let $\rho_k(i)$ be the distance to its $k$-th nearest neighbour in $P$ (leave-one-out) and $\nu_k(i)$ the distance to its $k$-th nearest neighbour in $Q$. Then

$$r_i = \frac{(n-1)\,\rho_k(i)^d}{m\,\nu_k(i)^d} \approx \frac{q(x_i)}{p(x_i)},$$

so $\frac{1}{n}\sum_i r_i^{1-s}$ estimates $\int p^s q^{1-s}$. The constant $B_{k,s} = \frac{\Gamma(k)^2}{\Gamma(k-s+1)\,\Gamma(k+s-1)}$ corrects the bias of the kNN ratio.

- `c_s_hat` estimates $C_s(P,Q)$.
- `c_s_prime_hat` estimates the derivative of $-\log C_s(P,Q)$. It is positive when $s > s^*$.

In [ ]:
def c_s_hat(P, Q, s, k=5):
    """kNN estimate of C_s(P,Q)."""
    P = P.cpu().numpy()
    Q = Q.cpu().numpy()
    n, d = P.shape
    m = Q.shape[0]
    rho = kth_dists(P, P, k, self_lo=True)   # k-NN distance within P (leave-one-out)
    nu  = kth_dists(P, Q, k, self_lo=False)  # k-NN distance from P samples to Q
    # Bias-correction constant Γ(k)² / (Γ(k-s+1) Γ(k+s-1)), computed in log space
    B = np.exp(2*gammaln(k) - gammaln(k - s + 1) - gammaln(k + s - 1))
    # (n-1) rho^d / (m nu^d) ≈ q(x)/p(x), so the mean estimates ∫ p^s q^(1-s)
    return -np.log(B * np.mean(((n - 1) * rho**d / (m * nu**d))**(1 - s)))

def c_s_prime_hat(P, Q, s, k=5):
    """kNN estimate of the derivative."""
    P = P.cpu().numpy()
    Q = Q.cpu().numpy()
    n, d = P.shape
    m = Q.shape[0]
    rho = kth_dists(P, P, k, self_lo=True)
    nu  = kth_dists(P, Q, k, self_lo=False)
    r_i = ((n - 1) * rho**d) / (m * nu**d)
    # Derivative of the log bias-correction term log B w.r.t. s
    G = digamma(k - s + 1) - digamma(k + s - 1) 
    # Derivative of log mean(r^(1-s)): a weighted average of -log r_i with weights r_i^(1-s)
    return G - np.sum(np.log(r_i) * r_i**(1-s)) / np.sum(r_i**(1-s))

Sanity check: at the true $s^*$, $\hat C_{s^*}$ should be close to $C(P,Q)$ and the derivative estimate should be close to 0.

In [ ]:
C_hat = c_s_hat(P, Q, true_s)
C_prime_hat = c_s_prime_hat(P, Q, true_s)
print(f"Estimates at true s* = {true_s:.4f}")
print(f"  C_hat       = {C_hat:.4f}   (true C = {true_c:.4f}, error = {C_hat - true_c:+.4f})")
print(f"  C_prime_hat = {C_prime_hat:+.4f}   (should be close to 0)")

### Bisection for $s^*$

Two variants of Nielsen's bisection on $s \in (0,1)$:
- **Finite difference**: compares $\hat C_{m-\delta}$ and $\hat C_{m+\delta}$ to decide which half contains the maximum.
- **Direct derivative**: uses the sign of the `c_s_prime_hat` estimator.

Both return the final $(\hat s^*, \hat C)$ plus the per-iteration traces.

In [ ]:
## Implements a finite difference approximation of the derivative

def nielsen_bisect_finite_difference(P, Q, iters=10, delta=1e-3):
    a, b = delta, 1 - delta
    s_trace, C_trace = [], []
    for _ in range(iters):
        m = 0.5 * (a + b)
        left  = c_s_hat(P, Q, m - delta)
        right = c_s_hat(P, Q, m + delta)
        # C_s is concave: if it is decreasing at m, the max lies in the left half
        if left > right:
            b = m
        else:
            a = m
        s = 0.5 * (a + b)
        C = c_s_hat(P, Q, s)
        s_trace.append(s)
        C_trace.append(C)
    s = s_trace[-1]
    C = C_trace[-1]
    return s, C, s_trace, C_trace


## Implements the D_s estimator for the direct derivative from the paper
def nielsen_bisect_direct_derivative(P, Q, iters=10):
    a, b = 0., 1.
    s_trace, C_trace = [], []
    for _ in range(iters):
        s = 0.5 * (a + b)
        C_prime = c_s_prime_hat(P, Q, s)
        # C_prime > 0  <=>  C_s decreasing at s  <=>  s* < s, so keep the left half
        if C_prime > 0:
            b = s
        else:
            a = s
        C = c_s_hat(P, Q, s)
        s_trace.append(s)
        C_trace.append(C)
    s = s_trace[-1]
    C = C_trace[-1]
    return s, C, s_trace, C_trace
    
# Plot the bisection trajectory of s and C against the true values
def plot_trace(s_trace, C_trace, s_true=0.5, C_true=None):
    it = np.arange(1, len(s_trace)+1)
    fig, ax = plt.subplots(1, 2, figsize=(10,4))

    ax[0].plot(it, s_trace, '-o')
    ax[0].axhline(s_true, linestyle='--')
    ax[0].set_title("s* estimate")
    ax[0].set_xlabel("iteration")
    ax[0].grid(alpha=0.3)

    ax[1].plot(it, C_trace, '-o')
    if C_true is not None:
        ax[1].axhline(C_true, linestyle='--')
    ax[1].set_title("Chernoff info")
    ax[1].set_xlabel("iteration")
    ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"plots/chernoff/{dataset_kwargs['name']}_1000samples.png")
    plt.show()

Run the direct-derivative bisection on the samples above and plot how $\hat s^*$ and $\hat C$ evolve over iterations (dashed lines = true values).

In [ ]:
s_est, C_est, s_trace, C_trace = nielsen_bisect_direct_derivative(P, Q)

In [ ]:
plot_trace(s_trace, C_trace, s_true=true_s, C_true=true_c)

### Convergence vs. sample size

Repeat the estimate `reps` times at each of several sample sizes (log-spaced from $10$ to $10^5$) to measure bias and variance. This cell is slow (~8 min).

In [ ]:
samples = np.logspace(1,5,8)  # sample sizes from 10 to 1e5
reps = 20                     # independent draws per sample size
derivative = []
for s in tqdm.tqdm(samples):
    s_draws, C_draws = [], []
    for _ in range(reps):
        P, Q, true_s, true_c = get_dataset(sample_size=int(s), **dataset_kwargs)
        s_der, C_der, _, _ = nielsen_bisect_direct_derivative(P, Q)
        s_draws.append(s_der)
        C_draws.append(C_der)
    # derivative[i] = (list of s estimates, list of C estimates) at samples[i]
    derivative.append((s_draws, C_draws))

In [ ]:
# Same sanity check on the last (largest) sample drawn in the loop above
C_hat = c_s_hat(P, Q, true_s)
C_prime_hat = c_s_prime_hat(P, Q, true_s)
print(f"Estimates at true s* = {true_s:.4f}")
print(f"  C_hat       = {C_hat:.4f}   (true C = {true_c:.4f}, error = {C_hat - true_c:+.4f})")
print(f"  C_prime_hat = {C_prime_hat:+.4f}   (should be close to 0)")

Plot mean estimates of $\hat s^*$ and $\hat C$ against sample size, with normal-approximation confidence bands.

In [ ]:
def mean_ci(draws, confidence=0.95):
    """Row-wise mean and half-width of a normal-approximation CI. draws: (n_sizes, reps)."""
    draws = np.asarray(draws)
    mean = draws.mean(axis=1)
    sem = draws.std(axis=1, ddof=1) / np.sqrt(draws.shape[1])
    z = norm.ppf(0.5 + confidence / 2)
    return mean, z * sem

def plot_comparison(samples, derivative, true_s, true_c, confidence=0.95):
    s_der, C_der = zip(*derivative)
    s_mean, s_ci = mean_ci(s_der, confidence)
    C_mean, C_ci = mean_ci(C_der, confidence)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(samples, s_mean, 's-', label='Derivative Estimator', color='darkorange')
    ax1.fill_between(samples, s_mean - s_ci, s_mean + s_ci, color='darkorange', alpha=0.2, label='95% CI')
    ax1.axhline(true_s, color='k', linestyle='--', label=f'True $s^*={true_s:.3f}$')
    ax1.set_xscale('log')
    ax1.set_xlabel('Sample size')
    ax1.set_ylabel('s')
    ax1.set_title('Optimal s')
    ax1.legend()

    ax2.plot(samples, C_mean, 's-', label='Derivative Estimator', color='darkorange')
    ax2.fill_between(samples, C_mean - C_ci, C_mean + C_ci, color='darkorange', alpha=0.2, label='95% CI')
    ax2.axhline(true_c, color='k', linestyle='--', label=f'True $C={true_c:.3f}$')
    ax2.set_xscale('log')
    ax2.set_xlabel('Sample size')
    ax2.set_ylabel('$\\hat{C}(P,Q)$')
    ax2.set_title('Chernoff Information')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"plots/chernoff/{list(dataset_kwargs.values())}.png")
    plt.show()

In [ ]:
plot_comparison(samples, derivative, true_s, true_c)

## MNIST Section

Estimate Chernoff information between real image classes. Images of digits 2 and 3 are projected to 32 dimensions with PCA (fit on both classes together). Two comparisons are run at each sample size:
- **(3 vs 2)**: $P$ = digit 3, $Q$ = digit 2. Distinct classes, so $\hat C$ should be clearly positive.
- **(3 vs 3)**: $P$, $Q$ = two disjoint subsamples of digit 3. A null baseline where the true $C = 0$ and $s^* = 1/2$.

In [ ]:
from sklearn.decomposition import PCA

def load_idx(path):
    """Read a raw MNIST IDX file: images -> (n, 784) float in [0,1], labels -> (n,) uint8."""
    with open(path, 'rb') as f:
        f.read(4); n = int.from_bytes(f.read(4), 'big')
        dims = [int.from_bytes(f.read(4), 'big') for _ in range(2)] if 'images' in path else []
        data = np.frombuffer(f.read(), np.uint8)
    return data.reshape(n, -1).astype(np.float32) / 255. if dims else data

imgs = load_idx('data/MNIST/raw/train-images-idx3-ubyte')
lbls = load_idx('data/MNIST/raw/train-labels-idx1-ubyte')
# Keep digits 2 and 3, project to 32 dims with a PCA fit on both classes jointly
X2, X3 = imgs[lbls == 2], imgs[lbls == 3]
pca = PCA(n_components=32).fit(np.vstack([X2, X3]))
X2, X3 = pca.transform(X2).astype(np.float32), pca.transform(X3).astype(np.float32)

# Sample sizes 10..1000; s_32/C_32 = digit 3 vs digit 2, s_33/C_33 = digit 3 vs digit 3 (null)
ns = np.unique(np.logspace(1, 3, 9).astype(int))
reps = 30
s_32, C_32 = np.zeros((len(ns), reps)), np.zeros((len(ns), reps))
s_33, C_33 = np.zeros((len(ns), reps)), np.zeros((len(ns), reps))

for i, n in enumerate(tqdm.tqdm(ns)):
    for r in range(reps):
        P = torch.tensor(X3[np.random.choice(len(X3), n, replace=False)])
        Q = torch.tensor(X2[np.random.choice(len(X2), n, replace=False)])
        s_32[i, r], C_32[i, r], _, _ = nielsen_bisect_direct_derivative(P, Q)
        # Two disjoint subsamples of the same class: true C = 0
        idx = np.random.choice(len(X3), 2 * n, replace=False)
        P2, Q2 = torch.tensor(X3[idx[:n]]), torch.tensor(X3[idx[n:]])
        s_33[i, r], C_33[i, r], _, _ = nielsen_bisect_direct_derivative(P2, Q2)

# NOTE: redefines mean_ci from above (fixed z=1.96, population std); later cells use this version
def mean_ci(a, confidence=0.95):
    a = np.asarray(a); m = a.mean(1)
    ci = 1.96 * a.std(1) / np.sqrt(a.shape[1])
    return m, ci

In [ ]:
# Left: estimated s* (null case should sit at 1/2). Right: estimated Chernoff information
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for a, lbl, c, mk, ax in [(s_32, '$\\hat{s}(2,3)$', 'darkorange', 's-', ax1),
                          (s_33, '$\\hat{s}(3,3)$', 'steelblue', 'o-', ax1)]:
    m, ci = mean_ci(a)
    ax.plot(ns, m, mk, label=lbl, color=c)
    ax.fill_between(ns, m - ci, m + ci, color=c, alpha=0.2)
ax1.axhline(0.5, color='black', linestyle='--', label='$s=1/2$')
ax1.set_xscale('log'); ax1.set_xlabel('Sample size'); ax1.set_ylabel('$\\hat{s}^*$')
ax1.set_title('Optimal s'); ax1.legend()

for a, lbl, c, mk in [(C_32, '$\\hat{C}(2,2)$', 'darkorange', 's-'),
                      (C_33, '$\\hat{C}(3,3)$', 'steelblue', 'o-')]:
    m, ci = mean_ci(a)
    ax2.plot(ns, m, mk, label=lbl, color=c)
    ax2.fill_between(ns, m - ci, m + ci, color=c, alpha=0.2)
ax2.set_xscale('log'); ax2.set_xlabel('Sample size'); ax2.set_ylabel('$\\hat{C}(P,Q)$')
ax2.set_title('Chernoff Information'); ax2.legend()

plt.tight_layout()
plt.savefig("plots/chernoff/MNIST.png")
plt.show()

## Dimensionality Scaling

kNN estimators degrade as dimension grows. Here the same synthetic family is generated in $d \in \{2,4,6,8\}$ and the absolute errors $|\hat s^* - s^*|$ and $|\hat C - C|$ are tracked against sample size.

In [ ]:
# For each dimension d, estimate (s*, C) across sample sizes and store with the true values
dims = [2, 4, 6, 8]
samples = np.logspace(1, 4, 8)
reps = 1
dim_results = {}
for d in dims:
    dataset_kwargs_d = {"name": "symmetric_truncnorm", "delta": 0.1, "sigma": 0.2, "dim": d}
    # dataset_kwargs_d = {"name": "symmetric_truncnorm", "lam1": [1.0] * d, "lam2": [8.0] * d}
    s_draws, C_draws = [], []
    for s in tqdm.tqdm(samples, desc=f"d={d}"):
        s_reps, C_reps = [], []
        for _ in range(reps):
            P, Q, true_s, true_c = get_dataset(sample_size=int(s), **dataset_kwargs_d)
            s_der, C_der, _, _ = nielsen_bisect_direct_derivative(P, Q)
            s_reps.append(s_der)
            C_reps.append(C_der)
        s_draws.append(s_reps)
        C_draws.append(C_reps)
    dim_results[d] = (s_draws, C_draws, true_s, true_c)

In [ ]:
def plot_dim_convergence(samples, dim_results):
    """Log-log plot of absolute error in s* and C vs. sample size, one line per dimension."""
    from matplotlib.colors import LinearSegmentedColormap
    seq_violet = LinearSegmentedColormap.from_list(
        "seq_violet", ["#c0bae0", "#857ac4", "#4a3aa7", "#3a2d82", "#29205c"]
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    colors = seq_violet(np.linspace(0, 1, len(dim_results)))

    for color, (d, (s_draws, C_draws, true_s, true_c)) in zip(colors, dim_results.items()):
        s_err = np.abs(np.array(s_draws) - true_s)
        C_err = np.abs(np.array(C_draws) - true_c)

        s_mean, s_ci = mean_ci(s_err)
        C_mean, C_ci = mean_ci(C_err)

        ax1.plot(samples, s_mean, 's-', color=color, label=f'd={d}')
        ax1.fill_between(samples, np.maximum(s_mean - s_ci, 1e-12), s_mean + s_ci, color=color, alpha=0.2)

        ax2.plot(samples, C_mean, 's-', color=color, label=f'd={d}')
        ax2.fill_between(samples, np.maximum(C_mean - C_ci, 1e-12), C_mean + C_ci, color=color, alpha=0.2)

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('Sample size')
    ax1.set_ylabel('$|\\hat{s}^* - s^*|$')
    ax1.set_title('Convergence of s estimate')
    ax1.legend()

    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel('Sample size')
    ax2.set_ylabel('$|\\hat{C}(P,Q) - C(P,Q)|$')
    ax2.set_title('Convergence of Chernoff Information estimate')
    ax2.legend()

    plt.tight_layout()
    plt.savefig("plots/chernoff/truncnorm_dims_convergence.png")
    plt.show()

In [ ]:
plot_dim_convergence(samples, dim_results)